In [ ]:
import pandas as pd
df = pd.read_csv('../Thesis_Data/Final_Data/Merged_Data_Scaled_Processed.csv')
df_sample = pd.read_csv('../Thesis_Data/Final_Data/sample_data.csv')


In [17]:
# Shape, farms, animals, records per animal distribution
df.groupby('Farm_Code')['Animal_ID'].nunique()

Farm_Code
521513     1304
600305      252
690616      506
861207     1231
1114232     552
           ... 
8701157     317
8710434     312
8900478      25
9330905     570
9410250     296
Name: Animal_ID, Length: 326, dtype: int64

In [12]:
df.groupby(['Farm_Code', 'Animal_ID']).size().describe()


count    88750.000000
mean         8.016879
std          4.051365
min          1.000000
25%          5.000000
50%          7.000000
75%         10.000000
max         33.000000
dtype: float64

In [29]:
mask = df.groupby(['Farm_Code', 'Animal_ID']).size()<6

len(df.groupby(['Farm_Code', 'Animal_ID']).size()[mask])

23151

In [30]:
counts = df.groupby(['Farm_Code', 'Animal_ID']).size()
for threshold in [3, 5, 6, 8, 10]:
    kept = (counts >= threshold).sum()
    pct = kept / len(counts) * 100
    print(f"≥{threshold} records: {kept} animals ({pct:.1f}%)")

≥3 records: 82177 animals (92.6%)
≥5 records: 72598 animals (81.8%)
≥6 records: 65599 animals (73.9%)
≥8 records: 44093 animals (49.7%)
≥10 records: 26098 animals (29.4%)


In [31]:
print("=== BEFORE ===")
print(f"Records:   {len(df):,}")
print(f"Animals:   {df['Animal_ID'].nunique():,}")
print(f"Farms:     {df['Farm_Code'].nunique():,}")
lactations_before = df.groupby('Animal_ID').ngroups
print(f"Lactations:{lactations_before:,}")

# Step 1 — Filter 2: farm-day density
daily_counts = df.groupby(['Farm_Code', 'dtt'])['Animal_ID'].nunique()
valid_days = daily_counts[daily_counts >= 5]
df = df.merge(valid_days.rename('day_count').reset_index(), on=['Farm_Code', 'dtt'])
df = df.drop(columns='day_count')

# Step 2 — Filter 1: lactation length
lac_counts = df.groupby(['Farm_Code', 'Animal_ID']).size()
valid_lacs = lac_counts[lac_counts >= 3].reset_index()[['Farm_Code', 'Animal_ID']]
df = df.merge(valid_lacs, on=['Farm_Code', 'Animal_ID'])

print("\n=== AFTER ===")
print(f"Records:   {len(df):,}")
print(f"Animals:   {df['Animal_ID'].nunique():,}")
print(f"Farms:     {df['Farm_Code'].nunique():,}")
lactations_after = df.groupby('Animal_ID').ngroups
print(f"Lactations:{lactations_after:,}")

=== BEFORE ===
Records:   711,498
Animals:   88,697
Farms:     326
Lactations:88,697

=== AFTER ===
Records:   691,589
Animals:   81,296
Farms:     315
Lactations:81,296


In [34]:
# Check the distribution of gaps between test dates per animal
df['dtt'] = pd.to_datetime(df['dtt'])

df = df.sort_values(['Farm_Code', 'Animal_ID', 'dtt'])

df['gap_days'] = df.groupby(['Farm_Code', 'Animal_ID'])['dtt'].diff().dt.days

print(df['gap_days'].describe())
print()
print("Gap distribution (days):")
print(df['gap_days'].value_counts(bins=[0,20,30,40,50,60,90,180,365]).sort_index())

count    610285.000000
mean         41.420563
std          32.261352
min           0.000000
25%          29.000000
50%          32.000000
75%          39.000000
max        1387.000000
Name: gap_days, dtype: float64

Gap distribution (days):
(-0.001, 20.0]      4459
(20.0, 30.0]      238237
(30.0, 40.0]      220866
(40.0, 50.0]       40471
(50.0, 60.0]       36247
(60.0, 90.0]       45716
(90.0, 180.0]      14598
(180.0, 365.0]      9228
Name: count, dtype: int64


In [35]:
# Check DIM column
print(df['DIM'].dtype)
print(df['DIM'].head(10))

object
6110      16 days 00:00:00
6111      51 days 00:00:00
6112     121 days 00:00:00
6113     149 days 00:00:00
6114     179 days 00:00:00
6115     213 days 00:00:00
6116     242 days 00:00:00
6117     269 days 00:00:00
9993      11 days 00:00:00
10149     11 days 00:00:00
Name: DIM, dtype: object


In [36]:
df['DIM'] = pd.to_timedelta(df['DIM']).dt.days
print(df['DIM'].describe())

count    691589.000000
mean        153.302928
std          99.535238
min           0.000000
25%          73.000000
50%         141.000000
75%         218.000000
max        1671.000000
Name: DIM, dtype: float64


In [40]:
# DIM should be between 5 and ~305 days for a standard lactation
print(df['DIM'].value_counts(bins=[0, 5, 50, 100, 150, 200, 250, 305, 400, 500]).sort_index())

# Any negative or zero DIM?
print("DIM <= 0:", (df['DIM'] <= 0).sum())
print("DIM > 365:", (df['DIM'] > 365).sum())

(-0.001, 5.0]       2094
(5.0, 50.0]       111631
(50.0, 100.0]     129054
(100.0, 150.0]    125823
(150.0, 200.0]    113204
(200.0, 250.0]     95799
(250.0, 305.0]     64672
(305.0, 400.0]     38052
(400.0, 500.0]      8689
Name: count, dtype: int64
DIM <= 0: 4
DIM > 365: 19383


In [41]:
# Drop invalid DIM
df = df[df['DIM'] > 0]          # removes 4 records
df = df[df['DIM'] <= 365]       # removes records beyond 1 year

print(f"Records remaining: {len(df):,}")
print(f"Animals remaining: {df['Animal_ID'].nunique():,}")

Records remaining: 672,202
Animals remaining: 81,257


In [42]:
counts = df.groupby(['Farm_Code', 'Animal_ID']).size()
valid = counts[counts >= 3].reset_index()[['Farm_Code', 'Animal_ID']]
df = df.merge(valid, on=['Farm_Code', 'Animal_ID'])

print(f"Final records: {len(df):,}")
print(f"Final animals: {df['Animal_ID'].nunique():,}")
print(f"Final farms:   {df['Farm_Code'].nunique():,}")

Final records: 672,009
Final animals: 81,144
Final farms:   315


In [43]:
import numpy as np
import pandas as pd
from scipy.sparse import lil_matrix, csr_matrix

# Get unique identifiers
animals = df[['Farm_Code', 'Animal_ID']].drop_duplicates().reset_index(drop=True)
farms   = df['Farm_Code'].unique()

n_animals = len(animals)          # 81,144  — bottom level
n_farms   = len(farms)            # 315
n_total   = 1
n_series  = n_animals + n_farms + n_total   # 81,460

# Index mappings
animal_to_idx = {row['Animal_ID']: i for i, row in animals.iterrows()}
farm_to_idx   = {f: n_animals + i for i, f in enumerate(farms)}
total_idx     = n_animals + n_farms   # last row

# Build S matrix (rows = all series, cols = bottom-level animals)
S = lil_matrix((n_series, n_animals), dtype=np.float32)

# Bottom level — identity block
for i in range(n_animals):
    S[i, i] = 1.0

# Farm level — each farm row sums its animals
for _, row in animals.iterrows():
    animal_idx = animal_to_idx[row['Animal_ID']]
    farm_idx   = farm_to_idx[row['Farm_Code']]
    S[farm_idx, animal_idx] = 1.0

# Total level — sums everything
S[total_idx, :] = 1.0

# Convert to CSR for efficient arithmetic
S = csr_matrix(S)

print(f"S matrix shape: {S.shape}")
print(f"Non-zero elements: {S.nnz:,}")
print(f"Density: {S.nnz / (S.shape[0] * S.shape[1]) * 100:.4f}%")

S matrix shape: (81468, 81152)
Non-zero elements: 243,456
Density: 0.0037%


In [48]:
S.shape

(81468, 81152)